In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.express as px
from itables import init_notebook_mode, show

client = bigquery.Client()

c:\Users\jy\anaconda3\envs\tzesm\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [2]:
query = """
    SELECT *
    FROM `tz-data-dev.taiwan_lnd.power_breakdown`
"""

# Run the query
query_job = client.query(query)

# Convert the result to a pandas DataFrame
df = query_job.to_dataframe()

c:\Users\jy\anaconda3\envs\tzesm\Lib\site-packages\google\cloud\bigquery\table.py:1933: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [7]:
df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M:%S')
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)
df['year'] = df.index.year
df['month'] = df.index.month
df['day'] = df.index.day
df['hour'] = df.index.hour

KeyError: 'datetime'

In [8]:
df_prod = df[['powerConsumptionTotal','powerProductionTotal', 'production_nuclear',
              'production_geothermal', 'production_biomass', 'production_coal',
              'production_wind', 'production_solar', 'production_hydro',
              'production_gas', 'production_oil', 'production_unknown',
              'production_hydro discharge', 'year', 'month', 'day', 'hour']]

In [ ]:
# df_prod_2024 = df_prod.loc['2024']
# demand = df_prod_2024['powerConsumptionTotal'].reset_index().drop(columns='datetime').rename(columns={'powerConsumptionTotal': 'TWN'})
# demand.to_csv('loads-p_set.csv', index=True)

In [49]:
def filter(df, month, day):
    mask = ~(
    (df['month'] == month) &
    (df['day']   == day)
    )

    return df[mask]

In [50]:
years = [2018, 2020, 2021, 2022, 2024]

In [51]:
df_demand_profile = df_prod[['powerConsumptionTotal', 'year', 'month', 'day', 'hour']].copy()

In [52]:
df_demand_profile = filter(df_demand_profile, 2, 29)

In [53]:
years = [2018, 2020, 2021, 2022, 2024]
df_selected_5Y = df_demand_profile[
    df_demand_profile['year'].isin(years)
]

In [68]:
df_selected_5Y_ave = df_selected_5Y.groupby(['month', 'day', 'hour']).mean().reset_index()
df_selected_5Y_ave = df_selected_5Y_ave[['powerConsumptionTotal']].rename(columns={'powerConsumptionTotal': '5Y-Average'})

In [72]:
df_selected_5Y_wide = df_selected_5Y.pivot_table(
    index=['month','day','hour'],
    columns='year',
    values='powerConsumptionTotal'
).reset_index(drop=True)

In [73]:
df_combined = pd.concat([df_selected_5Y_wide, df_selected_5Y_ave], axis=1)

In [75]:
px.line(
    df_combined,
    x=df_combined.index,
    y=df_combined.columns,
    title="Taiwan Power Consumption (2018, 2020, 2021, 2022, 2024, 5Y-Average)",
    labels={"index": "Hour", "value": "Power Consumption (MW)"}
).show()

In [86]:
df_selected_5Y_ave

,5Y-Average
0,22042.6
1,22627.2
2,23116.2
3,23404.8
4,22896.6
...,...
8755,20936.8
8756,20599.2
8757,20561.0
8758,20754.2


In [90]:
def normalised_demand_profile(df):
    """
    Normalise the demand profile to 1.
    """
    df = df.copy()
    total_demand = df['5Y-Average'].sum()
    df['5Y-Average'] = df['5Y-Average'] / total_demand
    return df

In [91]:
df_selected_5Y_ave_norm = normalised_demand_profile(df_selected_5Y_ave)

In [94]:
df_selected_5Y_ave_norm.to_csv('2030_Demand_Profile.csv', index=True)